In [ ]:
import sys
sys.path.append('..')
from osp import *


In [66]:
df_feats = pd.read_pickle('../data/raw/df_feats.pkl.gz')
df_feats.drop(columns=['weight','run'], inplace=True)
df_feats = df_feats.groupby('feature').mean(numeric_only=True).reset_index()
# df_feats['feature'] = df_feats['feature'].apply(lambda x: f'{FEAT2DESC[x]} ({x.split("_")[0]})')
df_feats = df_feats.set_index('feature')

In [92]:
phil_avg = df_feats[[c for c in df_feats.columns if 'Philosophy' in c]].mean(axis=1)
phil_avg

feature
deprel_acl            0.034133
deprel_acl:relcl      0.201338
deprel_advcl          0.135333
deprel_advcl:relcl    0.001412
deprel_advmod         0.236685
                        ...   
ttr_ADJ              -0.090164
ttr_ADV               0.022940
ttr_NOUN             -0.131978
ttr_VERB              0.014938
ttr_mean             -0.183295
Length: 99, dtype: float64

In [93]:
df_feats

,mean_1900-1925 Philosophy,mean_1925-1950 Philosophy,mean_1950-1975 Philosophy,mean_1975-2000 Philosophy,mean_2000-2025 Philosophy,mean_1900-1925 Literature,mean_1925-1950 Literature,mean_1950-1975 Literature,mean_1975-2000 Literature,mean_2000-2025 Literature,mean_1900-1925 Other,mean_1925-1950 Other,mean_1950-1975 Other,mean_1975-2000 Other,mean_2000-2025 Other
feature,,,,,,,,,,,,,,,
deprel_acl,-0.313101,-0.142648,0.045036,0.234638,0.346739,-0.549023,-0.565127,-0.500185,-0.328900,-0.249028,0.041986,0.087524,0.084107,0.039150,0.073456
deprel_acl:relcl,0.252283,0.159881,0.212713,0.220905,0.160908,-0.528456,-0.509101,-0.209944,0.070357,0.063475,-0.590179,-0.635209,-0.658398,-0.705738,-0.602884
deprel_advcl,-0.083547,-0.100643,0.219499,0.259378,0.381981,-0.572426,-0.498796,-0.345097,-0.231885,-0.210435,-0.471692,-0.445130,-0.476629,-0.404548,-0.306675
deprel_advcl:relcl,-0.217255,-0.218104,-0.012336,0.112319,0.342438,-0.223443,-0.277982,-0.241096,-0.142420,-0.073857,-0.355105,-0.345679,-0.319705,-0.217533,-0.057520
deprel_advmod,0.187026,0.080797,0.240140,0.308361,0.367103,-0.349919,-0.361730,-0.258166,-0.248061,-0.393468,-0.597761,-0.554387,-0.511372,-0.643990,-0.856087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ttr_ADJ,0.403454,0.195954,-0.131612,-0.336596,-0.582021,0.327089,0.506188,0.672838,0.590847,0.453015,0.278812,0.376645,0.367919,0.086685,-0.220280
ttr_ADV,0.293094,0.247926,0.010343,-0.134215,-0.302447,-0.112840,-0.034721,0.124955,0.207489,0.161909,0.195618,0.399960,0.395506,0.244974,0.065101
ttr_NOUN,0.460112,0.218416,-0.188604,-0.437737,-0.712079,0.531878,0.720212,0.752255,0.692206,0.712505,0.327860,0.451453,0.328928,0.039041,-0.193924


In [154]:
import pandas as pd
import numpy as np

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib as mpl

# Assume df_feats and phil_avg are already defined

# Optional: scale features (recommended for clustering)
X = StandardScaler().fit_transform(df_feats.values)

# Compute linkage (method can be "ward", "average", "complete", etc.)
Z = linkage(X, method="ward")

# Cut by distance threshold (clusters merge above this distance)
t = 5.0  # tune to get desired granularity; inspect Z[:, 2] for merge heights
cluster_labels = fcluster(Z, t, criterion="distance")

# Cut into k clusters (labels 1..k)
# k = 10  # set desired number of clusters
# cluster_labels = fcluster(Z, k, criterion="maxclust")

In [155]:
df_feat_clust = df_feats.copy()
df_feat_clust['cluster'] = cluster_labels
# df_feat_clust

In [156]:
df_feat_clust.cluster.max()

np.int32(17)

In [157]:
avgs = defaultdict(dict)
featclust = defaultdict(list)
featclust_str = defaultdict(list)

for c,cdf in df_feat_clust.groupby('cluster'):
    phil_avg_c = cdf[[x for x in cdf.columns if 'Philosophy' in x]].mean(axis=1)
    lit_avg_c = cdf[[x for x in cdf.columns if 'Literature' in x]].mean(axis=1)
    other_avg_c = cdf[[x for x in cdf.columns if 'Other' in x]].mean(axis=1)

    avgs[c]['phil'] = phil_avg_c.mean()
    avgs[c]['lit'] = lit_avg_c.mean()
    avgs[c]['other'] = other_avg_c.mean()

    # print(c, len(cdf))
    # print(f'Philosophy: {phil_avg_c.mean():.2f}')
    # print(f'Literature: {lit_avg_c.mean():.2f}')
    # print(f'Other: {other_avg_c.mean():.2f}')

    # print('\n'.join(cdf.index.to_list()))
    # print('\n------\n')
    featclust[c] = cdf.index.to_list()
    featclust_str[c] = '; '.join([
        f'{FEAT2DESC[x]} ({"+" if phil_avg[x] > 0 else ""}{phil_avg[x]:.2f})'
        for x in sorted(cdf.index.to_list(), key=lambda x: phil_avg[x], reverse=True)
    ])



In [158]:
pd.options.display.max_colwidth = 1000

avgs_df = pd.DataFrame(avgs).T.sort_values('phil', ascending=False).rename_axis('cluster')
avgs_df['phil_rank'] = avgs_df['phil'].rank(ascending=False)
avgs_df['lit_rank'] = avgs_df['lit'].rank(ascending=False)
avgs_df['other_rank'] = avgs_df['other'].rank(ascending=False)
avgs_df['num_feats'] = [len(featclust[c]) for c in avgs_df.index]
avgs_df['feats'] = [featclust[c] for c in avgs_df.index]
avgs_df['feats_str'] = [featclust_str[c] for c in avgs_df.index]
avgs_df.to_excel('../data/feat_clusts.xlsx')

In [ ]:
avgs_df2 = avgs_df.copy()
for c in ['phil', 'lit', 'other']:
    avgs_df2[c] = avgs_df2[c].round(2).apply(lambda x: f'{"+" if x > 0 else ""}{x}')
for c in ['phil_rank', 'lit_rank', 'other_rank']:
    avgs_df2[c] = avgs_df2[c].astype(int)
# avgs_df2.to_excel('../data/feat_clusts2.xlsx')
avgs_df2 = avgs_df2[['feats_str','phil', 'lit', 'other']]
avgs_df2 = avgs_df2.reset_index().rename(
    columns={
        'phil': 'Phil (z)', 'lit': 'Lit (z)', 'other': 'Other (z)',
        'phil_rank': 'Philosophy (rank)', 'lit_rank': 'Literature (rank)', 'other_rank': 'Other (rank)',
        'num_feats': '# Features', 'feats_str': 'Features', 'cluster': 'Cluster'
    })

In [167]:

avgs_df2

,Cluster,# Features,Features,Philosophy (z),Literature (z),Other (z)
0,2,4,Copula (+0.41); Modal (+0.34); Expletive (+0.31); Auxiliary (+0.28),+0.34,-0.49,-0.44
1,1,9,"Wh-determiner (+0.29); Verb, 3rd person sing. pres. (+0.27); Adverb (+0.25); Personal pronoun (+0.25); Adverbial modifier (+0.24); # Dependent clauses (+0.22); # Words in dependent clauses (+0.21); Relative clause modifier (+0.20); Nominal subject (+0.18)",+0.23,-0.3,-0.7
2,3,11,"Marker (+0.29); Verb, base form (+0.28); Verb, non-3rd person sing. pres. (+0.22); Existential there (+0.21); # Clause transitions (+0.21); Outer clause nominal subject (+0.19); Clausal complement (+0.18); Clausal subject (+0.16); # Unique clauses (+0.16); Adverbial clause modifier (+0.14); Maximum clause depth (+0.09)",+0.19,-0.47,-0.43
3,9,9,"Preposition or subordinating conjunction (+0.22); Predeterminer (+0.16); Predeterminer (+0.16); Determiner (+0.11); Clausal passive subject (+0.10); Determiner (+0.07); Adjective, superlative (+0.03); Adjective, comparative (-0.01); Oblique agent in passive construction (-0.03)",+0.09,-0.03,+0.1
4,8,3,"Passive auxiliary (+0.16); Passive nominal subject (+0.10); Verb, past participle (-0.03)",+0.08,-0.13,+0.66
5,15,9,to (+0.15); Wh-pronoun (+0.10); Fixed multiword expression (+0.09); Open clausal complement (+0.09); Outer clause clausal subject (+0.06); Overridden disfluency (+0.04); Preconjunct (+0.02); Wh-adverb (+0.00); Adverbial relative clause modifier (+0.00),+0.06,-0.16,-0.23
6,6,5,Coordinating conjunction (+0.11); Type-token ratio for adverbs (+0.02); Type-token ratio for verbs (+0.01); Nominal modifier (+0.01); Case marking (-0.06),+0.02,+0.18,+0.22
7,16,6,"Adjective (+0.13); Clausal modifier of noun (adnominal clause) (+0.03); Noun, singular or mass (+0.03); Maximum word depth (-0.03); Adjectival modifier (-0.07); Noun, plural (-0.08)",0.0,-0.37,+0.23
8,12,7,Superfluous punctuation (-0.01); Goes with (-0.01); Discourse element (-0.02); 'Goes with' or error marker (-0.02); Detached prefix or suffix (-0.02); Symbol (-0.04); Hyphen (-0.09),-0.03,-0.07,+0.04
9,10,10,"Orphan (+0.00); Unmarked nominal modifier (-0.00); Oblique nominal (-0.01); Dislocated elements (-0.01); Adverb, comparative (-0.02); Possessive wh-pronoun (-0.03); Adverb, superlative (-0.05); Particle (-0.07); Phrasal verb particle (-0.07); Unmarked oblique nominal (-0.11)",-0.04,+0.12,+0.03


In [168]:
out = df_to_latex_table(avgs_df2)
with open('../../../Dropbox/Prof/Articles/OSP/tables/feat_clusts.tex', 'w') as f:
    f.write(out)